# 02 — Facial Image Classifier

Builds the facial image branch of the multimodal ASD diagnosis system using 
EfficientNet-B0. Produces both a standalone ASD classifier and a feature 
extractor whose 1280-dimensional embeddings feed into the fusion model.

## What this notebook does
- Builds a dataset manifest from YOLO-format images with remapped binary labels
- Defines a custom PyTorch Dataset with augmentation for training
- Trains EfficientNet-B0 with weighted cross-entropy loss and early stopping
- Evaluates on the held-out test set and reports clinical metrics

## Key finding
Two-phase transfer learning (freeze → unfreeze) was tested and failed on this 
clinical domain. Single-phase full fine-tuning at lr=1e-4 achieved 86% accuracy 
and 0.83 ASD recall. Both approaches are documented here.

## Output
`best_face_model.pth` — trained EfficientNet-B0 weights (best validation checkpoint)

In [ ]:
import os
import pandas as pd
from pathlib import Path
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
from dotenv import load_dotenv
import wandb
from PIL import Image
import torchvision.models as models
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import torch.optim as optim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Checking the path
BASE_PATH = '/content/drive/MyDrive/Dataset'
for split in ['train', 'val', 'test']:
    path = f'{BASE_PATH}/autism_dataset/images/{split}'
    files = os.listdir(path)[:5]
    print(f"{split}: {files}")

In [ ]:
# Paths
FACE_BASE  = f'{BASE_PATH}/autism_dataset'

# images/ and labels/ share the same split subfolder structure
IMG_BASE   = f'{FACE_BASE}/images'   
LBL_BASE   = f'{BASE_PATH}/autism_dataset/images'  

SPLITS = ['train', 'val', 'test']

# YOLO → Binary class remap ─
# YOLO class 0 (autistic_face) → 1 (ASD)
# YOLO class 1 (autism)        → 1 (ASD)
# YOLO class 2 (no_autism)     → 0 (TD)
YOLO_REMAP = {0: 1, 1: 1, 2: 0}



# label parser
def parse_yolo_label(label_path):
    """
    Reads a YOLO .txt label file and returns the remapped binary class.
    
    Takes the first bounding box line only.
    Returns None if the file is empty or unreadable.
    """
    try:
        with open(label_path, 'r') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
        if not lines:
            return None  # Empty label file — skip
        class_id = int(lines[0].split()[0])  # First token of first line
        return YOLO_REMAP.get(class_id, None)  # None if unexpected class
    except Exception:
        return None
    

# Scan all splits
records = []

for split in SPLITS:
    lbl_dir = Path(LBL_BASE) / split
    img_dir = Path(IMG_BASE) / split

    label_files = list(lbl_dir.glob('*.txt'))
    print(f"\n[{split.upper()}] Found {len(label_files)} label files")

    skipped_no_image   = 0
    skipped_bad_label  = 0

    for lbl_path in label_files:
        # Derive corresponding image path (try .jpg then .png)
        stem = lbl_path.stem
        img_path = img_dir / f'{stem}.jpg'
        if not img_path.exists():
            img_path = img_dir / f'{stem}.png'
        if not img_path.exists():
            skipped_no_image += 1
            continue

        label = parse_yolo_label(lbl_path)
        if label is None:
            skipped_bad_label += 1
            continue

        records.append({
            'image_path': str(img_path),
            'label': label,           # 0 = TD, 1 = ASD
            'label_name': 'ASD' if label == 1 else 'TD',
            'split': split
        })

    print(f"  Skipped — no matching image: {skipped_no_image}")
    print(f"  Skipped — empty/bad label:   {skipped_bad_label}")


# build the dataframe
df = pd.DataFrame(records)
print(f"Total samples loaded: {len(df)}")
# df.head()

In [ ]:
# Class distribution
dist = df.groupby(['split', 'label_name']).size().unstack(fill_value=0)
print(dist)

In [ ]:
print(f"Unique labels present:        {sorted(df['label'].unique())}")
print(f"Any nulls in image_path:      {df['image_path'].isna().sum()}")
print(f"Any nulls in label:           {df['label'].isna().sum()}")
print(f"Duplicate image paths:        {df['image_path'].duplicated().sum()}")

In [ ]:
# Verify a few image paths actually exist on disk
sample_check = df.sample(5, random_state=42)
for _, row in sample_check.iterrows():
    exists = Path(row['image_path']).exists()
    print(f"  {'' if exists else 'MISSING'} {row['image_path']}")

In [ ]:
# Save to own MyDrive (dataset drive is read-only)
SAVE_PATH = '/content/drive/MyDrive/face_dataset_manifest.csv'  
df.to_csv(SAVE_PATH, index=False)
print(f"\n Manifest saved → {SAVE_PATH}")
print(f"   Shape: {df.shape}")

In [ ]:
manifest = pd.read_csv(SAVE_PATH)
manifest['split'].value_counts()

## Custom PyTorch Dataset Class

In [ ]:
# Define transform
# ImageNet stats
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE      = 224

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, manifest_df, split, transform=None):
        self.data      = manifest_df[manifest_df['split'] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = torch.tensor(row['label'], dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# Instantiate datasets
manifest = pd.read_csv(SAVE_PATH)

train_dataset = FaceDataset(manifest, 'train', transform=train_transforms)
val_dataset   = FaceDataset(manifest, 'val',   transform=val_test_transforms)
test_dataset  = FaceDataset(manifest, 'test',  transform=val_test_transforms)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# Sample
img_tensor, label = train_dataset[0]

print(f"Image tensor shape : {img_tensor.shape}")   # Expected: torch.Size([3, 224, 224])
print(f"Label              : {label.item()}")        # Expected: 0 or 1
print(f"Pixel min/max      : {img_tensor.min():.3f} / {img_tensor.max():.3f}")  # Normalized range

## DataLoaders + Class Weights

In [ ]:
# Create dataloaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

In [ ]:
# Compute class weights for imbalanced dataset
train_labels = manifest[manifest['split'] == 'train']['label'].tolist()
counts       = Counter(train_labels)           # {0: 1150, 1: 1312}
total        = sum(counts.values())            # 2462
num_classes  = 2

weights = [total / (num_classes * counts[i]) for i in range(num_classes)]
class_weights = torch.tensor(weights, dtype=torch.float)

print(f"Class counts  — TD: {counts[0]} | ASD: {counts[1]}")
print(f"Class weights — TD: {class_weights[0]:.4f} | ASD: {class_weights[1]:.4f}")

In [ ]:
# Verify batch
images, labels = next(iter(train_loader))

print(f"Batch image shape : {images.shape}")    # Expected: torch.Size([32, 3, 224, 224])
print(f"Batch label shape : {labels.shape}")    # Expected: torch.Size([32])
print(f"Labels in batch   : {labels.tolist()}")

## EfficientNet model setup

In [ ]:
# Load pretrained EfficientNet-B0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = models.efficientnet_b0(weights='IMAGENET1K_V1')
print("Pretrained EfficientNet-B0 loaded")

In [ ]:
# Replace classifier head
# EfficientNet-B0 outputs 1280 features before the classifier
EMBEDDING_DIM = 1280

model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(EMBEDDING_DIM, 2)
)

model = model.to(device)
print("Classifier head replaced — output: 2 classes")
print(f"New classifier: {model.classifier}")

In [ ]:
# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Classifier stays unfrozen
for param in model.classifier.parameters():
    param.requires_grad = True

# Verify
frozen     = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Frozen params    : {frozen:,}")
print(f"Trainable params : {trainable:,}")

In [ ]:
# Quick forward pass sanity check
model.eval()
with torch.no_grad():
    sample_imgs, sample_labels = next(iter(train_loader))
    sample_imgs = sample_imgs.to(device)
    output = model(sample_imgs)

print(f"Input shape  : {sample_imgs.shape}")
print(f"Output shape : {output.shape}")    # Expected: torch.Size([32, 2])
print(f"Sample logits: {output[0]}")       # Raw scores for TD and ASD

## Experiment 1 — Two-Phase Training (Attempted, Failed)

Two-phase transfer learning was tested first: freeze the backbone and train 
only the classifier head (Phase 1), then unfreeze and fine-tune the full 
network (Phase 2). This is standard practice for transfer learning.

**Result:** Phase 1 validation accuracy stuck at 41–43% — near random. 
The ImageNet-to-clinical-ASD domain gap is too large for frozen features 
to bridge. See Experiment 2 for the approach that worked.

In [ ]:
# Weighted loss — accounts for class imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Only optimize trainable params (classifier head for now)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

print(f"Loss    : CrossEntropyLoss with weights {class_weights}")
print(f"Optimizer: Adam | lr=1e-3")

In [ ]:
# Training and validation functions
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)

            total_loss += loss.item()
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total

### NOTE: This run produced ~41% val accuracy. See Experiment 2 below.

In [ ]:
# Training (frozen backbone, 5 epochs)

PHASE1_EPOCHS = 5
CKPT_PATH     = '/content/drive/MyDrive/best_face_model.pth' 

best_val_loss = float('inf')
history       = []

print("Phase 1: Training classifier head only\n")

for epoch in range(1, PHASE1_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    history.append({
        'epoch': epoch, 'phase': 1,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc
    })

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT_PATH)
        saved = ' saved'
    else:
        saved = ''

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} {saved}")

In [ ]:
for param in model.features.parameters():
    param.requires_grad = True

# Lower learning rate — backbone weights are delicate
optimizer = optim.Adam(model.parameters(), lr=1e-4)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone unfrozen | Trainable params: {trainable:,}")
print(f"Optimizer reset: Adam | lr=1e-4")

In [ ]:
# Training full network (10 epochs)
PHASE2_EPOCHS = 10

print("Phase 2: Fine-tuning full network \n")

for epoch in range(PHASE1_EPOCHS + 1, PHASE1_EPOCHS + PHASE2_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    history.append({
        'epoch': epoch, 'phase': 2,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CKPT_PATH)
        saved = 'saved'
    else:
        saved = ''

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} {saved}")

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv('/content/drive/MyDrive/face_training_history.csv', index=False)
print(history_df.to_string(index=False))

## Training — Single Phase Full Fine-tuning

Full network fine-tuned from the start at lr=1e-4. The low learning rate 
protects pretrained weights without freezing, achieving the same goal as 
Phase 1 freezing while allowing full backbone adaptation from epoch 1.

In [ ]:
# Login
load_dotenv('/content/drive/MyDrive/.env')
WANDB_API_KEY = os.getenv('WANDB_API_KEY')


wandb.login(key=WANDB_API_KEY)
print(" WandB authenticated")

In [ ]:
# Loss, optimizer and scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
print(f"Criterion ready | weights device: {class_weights.to(device).device}")

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print(f"Loss      : CrossEntropyLoss | weights {class_weights}")
print(f"Optimizer : Adam | lr=1e-3")
print(f"Scheduler : ReduceLROnPlateau | factor=0.5 | patience=2")

In [ ]:
# Early stopping class
class EarlyStopping:
    def __init__(self, patience=5, ckpt_path='/content/drive/MyDrive/best_face_model.pth'): 
        self.patience   = patience
        self.ckpt_path  = ckpt_path
        self.best_loss  = float('inf')
        self.counter    = 0
        self.stop       = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.ckpt_path)
            return True   # improved
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
            return False  # no improvement

early_stopping = EarlyStopping(patience=5)
print(f"Early stopping: patience=5 | checkpoint → {early_stopping.ckpt_path}")

In [ ]:
# Reset model to pretrained weights before retraining
# Fresh model — avoid inheriting overfitted weights from first run
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(EMBEDDING_DIM, 2)
)
model = model.to(device)

# Freeze backbone for Phase 1
for param in model.features.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Fresh model ready | Trainable params: {trainable:,}")

In [ ]:
# WandB initialization with config
run = wandb.init(
    entity="mbaraka-carnegie-mellon-university",
    project="Multimodal-AI-Fusion-Project",
    name="efficientnet-b0-face-branch",
    config={
        "architecture"  : "EfficientNet-B0",
        "dataset"       : "ASD Facial Images",
        "total_images"  : 3398,
        "img_size"      : 224,
        "batch_size"    : 32,
        "phase1_epochs" : 5,
        "phase2_max_epochs": 30,
        "phase1_lr"     : 1e-3,
        "phase2_lr"     : 1e-4,
        "dropout"       : 0.3,
        "early_stopping_patience": 5,
        "scheduler_patience"     : 2,
        "scheduler_factor"       : 0.5,
        "class_weights" : {"TD": round(class_weights[0].item(), 4), 
                           "ASD": round(class_weights[1].item(), 4)}
    }
)
print(f"WandB run initialized: {run.name}")

In [ ]:
# Training
PHASE1_EPOCHS = 5
history       = []

print("Phase 1: Classifier head only\n")

for epoch in range(1, PHASE1_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_loss)
    saved = early_stopping.step(val_loss, model)

    history.append({
        'epoch': epoch, 'phase': 1,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc
    })

    run.log({
        'epoch'      : epoch,
        'phase'      : 1,
        'train_loss' : train_loss,
        'train_acc'  : train_acc,
        'val_loss'   : val_loss,
        'val_acc'    : val_acc,
        'lr'         : optimizer.param_groups[0]['lr']
    })

    flag = 'saved' if saved else ''
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} {flag}")

In [ ]:
model.train()
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

outputs = model(images)
loss    = criterion(outputs, labels)

print(f"Output shape : {outputs.shape}")
print(f"Sample output: {outputs[0]}")
print(f"Loss         : {loss.item()}")
print(f"Labels sample: {labels[:8]}")

In [ ]:
# Unfreeze backbone for Phase 2
for param in model.features.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)
early_stopping = EarlyStopping(patience=5)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone unfrozen | Trainable params: {trainable:,}")
print(f"Optimizer reset: Adam | lr=1e-4")

In [ ]:
# Phase 2 full network fine-tuning
PHASE2_MAX_EPOCHS = 30

print("Phase 2: Full network fine-tuning \n")

for epoch in range(PHASE1_EPOCHS + 1, PHASE1_EPOCHS + PHASE2_MAX_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_loss)
    saved = early_stopping.step(val_loss, model)

    history.append({
        'epoch': epoch, 'phase': 2,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,     'val_acc': val_acc
    })

    run.log({
        'epoch'      : epoch,
        'phase'      : 2,
        'train_loss' : train_loss,
        'train_acc'  : train_acc,
        'val_loss'   : val_loss,
        'val_acc'    : val_acc,
        'lr'         : optimizer.param_groups[0]['lr']
    })

    flag = 'saved' if saved else f'no improvement ({early_stopping.counter}/{early_stopping.patience})'
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | {flag}")

    if early_stopping.stop:
        print(f"\n⏹ Early stopping triggered at epoch {epoch}")
        break

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv('/content/drive/MyDrive/face_training_history.csv', index=False) 
print(history_df.to_string(index=False))

In [ ]:
# Log best results summary to WandB
run.summary['best_val_loss'] = early_stopping.best_loss
run.summary['best_epoch']    = history_df.loc[history_df['val_loss'].idxmin(), 'epoch']
run.summary['best_val_acc']  = history_df.loc[history_df['val_loss'].idxmin(), 'val_acc']

run.finish()
print("WandB run complete")

## Evaluation

In [ ]:
# Load best checkpoint into model
model.load_state_dict(torch.load('/content/drive/MyDrive/best_face_model.pth'))
model.eval()
print(" Best checkpoint loaded (epoch 8)")

In [ ]:
# Run inference on test set
all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds   = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(f"Inference complete | Total samples: {len(all_preds)}")

In [ ]:
# Classification report
print(classification_report(
    all_labels, all_preds,
    target_names=['TD (0)', 'ASD (1)']
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['TD', 'ASD'])
disp.plot(cmap='Blues')
plt.title('EfficientNet-B0 — Test Set Confusion Matrix')
plt.savefig('/content/drive/MyDrive/face_confusion_matrix.png', dpi=150, bbox_inches='tight') 
plt.show()
print("Saved → face_confusion_matrix.png")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
ax1.plot(history_df['epoch'], history_df['train_loss'], label='Train Loss')
ax1.plot(history_df['epoch'], history_df['val_loss'],   label='Val Loss')
ax1.axvline(x=8, color='red', linestyle='--', label='Best epoch (8)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()


# Accuracy curves
ax2.plot(history_df['epoch'], history_df['train_acc'], label='Train Acc')
ax2.plot(history_df['epoch'], history_df['val_acc'],   label='Val Acc')
ax2.axvline(x=8, color='red', linestyle='--', label='Best epoch (8)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training vs Validation Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/face_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → face_training_curves.png")

## Results Summary

| Metric | Value |
|---|---|
| Test Accuracy | 86.0% |
| ASD Recall | 0.83 |
| ASD Precision | 0.77 |
| ASD F1 | 0.80 |
| TD Recall | 0.88 |
| False Negatives | 27 / 160 ASD cases missed |

The face-only model correctly identifies 83% of ASD children from a facial 
photograph. This establishes the unimodal baseline the fusion model must beat.